In [1]:
with open('../../config/xfinlink_api.key', 'r') as f:
    XFINLINK_API_KEY = f.readlines()[0].strip()

In [2]:
import xfinlink as xfl

xfl.set_api_key(XFINLINK_API_KEY)

### Price Data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

spx_cons = pd.read_csv(
    'sp500-constituents-2026-11-16.csv',
    encoding = 'cp949'
)

In [5]:
df = xfl.prices(
    tickers = spx_cons['Ticker'].tolist(),
    period = 'max'
)

/Users/tommy/miniconda3/envs/conda_env/lib/python3.13/site-packages/xfinlink/client.py:1121: UserWarning: xfinlink: 5 batch(es) failed: CHTR,CVX,CMG...: xfinlink: Rate limit reached (Daily limit reached (10,000 requests/day). Resets at midnight UTC.). Fix: wait 28092s, or reduce requests by adding start/end date filters and using max_rows parameter. The 165000 rows from the 33 page(s) fetched before this error are on the exception as .partial.; MAR,MRSH,MLM...: xfinlink: Rate limit reached (Daily limit reached (10,000 requests/day). Resets at midnight UTC.). Fix: wait 28091s, or reduce requests by adding start/end date filters and using max_rows parameter.; FTNT,FTV,FOXA...: xfinlink: Rate limit reached (Daily limit reached (10,000 requests/day). Resets at midnight UTC.). Fix: wait 28091s, or reduce requests by adding start/end date filters and using max_rows parameter. The 160000 rows from the 32 page(s) fetched before this error are on the exception as .partial.; MMM,AES,APA...: xfin

In [34]:
res = pd.DataFrame(df.set_index(['ticker','entity_name','gics_sector','date','exchange_code','price_basis','shares_as_of'])[
    [
        'open','high','low','close','adj_close','volume','shares_outstanding','split_ratio','dividend'
    ]
].stack().dropna())

In [35]:
res.index.names = [
    'ticker', 'entity_name', 'gics_sector', 'date', 'exchange_code', 'price_basis', 'shares_as_of', 'item'
]

In [36]:
res.columns = ['value']

In [38]:
res.to_parquet('xfinlink_sample.parquet')

In [43]:
res.index.get_level_values(0).unique()

Index(['ZBH', 'ZBRA', 'ZMH', 'ZTS'], dtype='str', name='ticker')

#### Fundamental Data

In [44]:
fund = xfl.fundamentals(
    tickers = spx_cons['Ticker'].tolist()[:100],
    period = 'max',
    period_type = 'quarterly'
)

/var/folders/hg/hwkx0k3x6ts_j4lz6kx68vd00000gn/T/ipykernel_35752/1636277946.py:1: UserWarning: xfinlink: Results truncated at max_rows (10000 rows). Later tickers in the request may be missing entirely. Fix: narrow your query with start/end dates or fewer tickers, or raise max_rows.
  fund = xfl.fundamentals(


In [56]:
tmp = spx_cons['Ticker'].tolist()[:100]

unique_tickers = set(tmp).difference(set(list(fund['ticker'].unique())))

In [59]:
unique_tickers = list(unique_tickers)

In [47]:
list(fund['ticker'].unique())

<ArrowStringArray>
[ 'AAPL',   'ABT',  'ADBE',   'ADI',   'ADM',   'ADP',  'ADSK',   'AEE',
   'AEP',   'AFL',   'AIG',  'AMAT',   'AMD',   'AME',  'AMGN',   'AON',
   'APA',   'APD',   'APH',   'ATO',   'AVY',   'AXP',    'BA',   'BAC',
  'BALL',   'BAX',   'BBY',   'BDX',  'BF-B',   'BKR',   'BMY',   'BNY',
 'BRK-B',   'CAH',  'CASY',   'CAT',   'CCL',  'CDNS',   'CMS',   'CNP',
   'CSX',   'CVS',   'LNT',   'MMM',    'MO',     'T']
Length: 46, dtype: str

In [60]:
from tqdm import tqdm

res_list = []

for ticker in tqdm(unique_tickers) :
    tmp_df = xfl.fundamentals(
        ticker = ticker,
        period = 'max',
        period_type = 'quarterly'
    )
    res_list.append(tmp_df)

comb_df = pd.concat(res_list, axis = 0)

100%|██████████| 54/54 [02:16<00:00,  2.54s/it]


In [71]:
res_list2 = []

for ticker in tqdm(spx_cons['Ticker'].tolist()[100:]) :
    try:
        tmp_df = xfl.fundamentals(
            ticker = ticker,
            period = 'max',
            period_type = 'quarterly'
        )
        res_list2.append(tmp_df)
    except :
        print(ticker)
        continue

comb_df2 = pd.concat(res_list2, axis = 0)

 95%|█████████▍| 382/403 [11:49<00:34,  1.63s/it]

VYLR


100%|██████████| 403/403 [12:27<00:00,  1.85s/it]


In [73]:
res_fund = pd.concat(
    [
        fund,
        comb_df,
        comb_df2
    ], axis = 0
)

In [74]:
res_fund.to_parquet(
    'xfinlink_fund_sample.parquet'
)